In [21]:
# Install required packages if not already installed
!pip install matplotlib seaborn wordcloud

In [23]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ipykernel
from wordcloud import WordCloud

df = pd.read_csv('perfumes_table.csv')

In [2]:
# Display the first few rows of the dataframe
df.head()

In [3]:
# Display basic statistics of the dataframe
df.describe()

In [4]:
#get rid of na in notes column
df = df.dropna(subset=['notes'])
#get rid of empty strings in notes column
df = df[df['notes'] != '']

In [5]:
#best rated perfumes title and rating
df_best_rated = df[df['rating'] == 5]
df_best_rated[['title', 'rating']]
# Display the best rated perfumes
df_best_rated

Brands

In [6]:
#best rated brands

# Calculate statistics per brand
brand_stats = df.groupby('designer')['rating'].agg(
    perfume_count='count',  # Number of perfumes per brand
    average_rating='mean',  # Average rating per brand
    min_rating='min',       # Minimum rating per brand
    max_rating='max'        # Maximum rating per brand
).reset_index()

# Sort by average rating (descending order)
brand_stats_sorted = brand_stats.sort_values(by='average_rating', ascending=False)

# Display the top 10 best-rated brands with perfume count and ratings
print("Top 10 Best-Rated Brands with Perfume Count and Ratings:")
print(brand_stats_sorted.head(20))

# Save the results to a CSV file (optional)
brand_stats_sorted.to_csv('brand_stats.csv', index=False)


In [7]:
brands_with_multiple_perfumes = brand_stats[brand_stats['perfume_count'] > 1]
brands_with_multiple_perfumes.sort_values(by='average_rating', ascending=False).head(10)

In [8]:
# Top perfume brands (if there's a 'brand' column)
df['designer'] = df['designer'].str.replace('perfumes and colognes', '', regex=True)
df['designer'] = df['designer'].str.strip()
top_brands = df['designer'].value_counts().head(20)

plt.figure(figsize=(10, 5))
sns.barplot(x=top_brands.values, y=top_brands.index, palette='viridis')
plt.title("Top 10 Most Common Perfume Brands")
plt.xlabel("Count")
plt.ylabel("Perfume Brands")
plt.show()

Notes

In [9]:
# Extract the 'notes' column from the DataFrame
notess = df['notes'].dropna().str.replace(r"\[|\]|'", "", regex=True).str.split(',').explode().str.strip()

# 1. Count the number of unique notes
unique_notes_count = notess.nunique()
print(f"Number of unique notes: {unique_notes_count}")

# 2. Frequency of each note
note_frequencies = notess.value_counts().reset_index()
note_frequencies.columns = ['Note', 'Frequency']
print("\nFrequency of each note:")
print(note_frequencies)

In [10]:
# Top perfume notes 
if 'notes' in df.columns:
    df['notes'] = df['notes'].str.replace("\[|\]|'", "", regex=True)
    notess = df['notes'].dropna().str.split(',').explode()
    notess = notess.str.strip()
    most_notes = notess.value_counts().head(20)

    plt.figure(figsize=(10, 5))
    sns.barplot(x=most_notes.values, y=most_notes.index, palette='viridis')
    plt.title("Top 20 Most Common Perfume Notes")
    plt.xlabel("Count")
    plt.ylabel("Perfume Notes")
    plt.show()


In [11]:
#unique notes

notess = df['notes'].dropna().str.split(',').explode()
notess = notess.str.strip()
least_notes = notess.value_counts().tail(20)

least_notes_df = least_notes.reset_index()
least_notes_df.columns = ['Perfume Note', 'Count']  # Rename columns
least_notes

#display the notes with only one occurence
note_counts = notess.value_counts()
unique_notes = note_counts[note_counts == 1].reset_index()
unique_notes.columns = ['Perfume Note', 'Count']
unique_notes



In [24]:

# Clean and process the notes column
df['notes'] = df['notes'].str.replace(r"\[|\]|'", "", regex=True)  # Remove brackets and apostrophes
notess = df['notes'].dropna().str.split(',').explode()
notess = notess.str.strip()

# Reset the index of the original DataFrame
df_reset = df.reset_index(drop=True)

# Merge the exploded notes back with the original DataFrame
df_exploded = df_reset.assign(notes=notess.reset_index(drop=True)).dropna(subset=['notes'])

# Calculate average rating per note
note_ratings = df_exploded.groupby('notes')['rating'].agg(
    average_rating='mean',  # Average rating per note
    count='count'           # Number of perfumes containing the note
).reset_index()


# Sort by average rating (descending order)
top_rated_notes = note_ratings.sort_values(by='average_rating', ascending=False)

# Filter for notes that appear more than once
note_ratings_filtered = note_ratings[note_ratings['count'] > 1]

# Sort by average rating (descending order)
top_rated_notes = note_ratings_filtered.sort_values(by='average_rating', ascending=False)

# Display the top-rated notes
print("Top-Rated Notes (with count > 1):")
print(top_rated_notes.head(20))  # Display top 20



In [13]:
# Create a word cloud of perfume descriptions
wordcloud = WordCloud(width=800, height=400, background_color='white').generate(' '.join(df['description'].dropna()))
plt.figure(figsize=(10, 5))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title("Word Cloud of Perfume Descriptions")
plt.show()


In [14]:
#Get rid of common words in the descriptions
def preprocess_text(text):
    # List of words to remove
    stop_words = {"top note", "launched", "middle note", "base note", "fragrance", "Top Note", "Middle Note", "Base Note", "notes", "fragrances", "fragrance", "perfume", "cologne", "women", "men", "Top", "nose behind"}
    # Split the text into words, filter out unwanted words, and join back into a single string
    return ' '.join(word for word in text.split() if word.lower() not in stop_words)

# Apply preprocessing to the 'description' column
cleaned_descriptions = df['description'].dropna().apply(preprocess_text)
wordcloud1 = WordCloud(width=800, height=400, background_color='white').generate(' '.join(cleaned_descriptions))

# Plot the word cloud
plt.figure(figsize=(10, 5))
plt.imshow(wordcloud1, interpolation='bilinear')
plt.axis('off')
plt.title("Word Cloud of Perfume Descriptions (Cleaned)")
plt.show()

In [15]:
#create a wordcloud for the reviews
wordcloudrev = WordCloud(width=800, height=400, background_color='white').generate(' '.join(df['reviews'].dropna()))
plt.figure(figsize=(10, 5))
plt.imshow(wordcloudrev, interpolation='bilinear')
plt.axis('off')

plt.title("Word Cloud of Perfume Reviews")
plt.show()



In [16]:
def preprocess(text):
    # List of words to remove
    stop_words = {"smell", "launched", "fragrance", "one"}
    # Split the text into words, filter out unwanted words, and join back into a single string
    return ' '.join(word for word in text.split() if word.lower() not in stop_words)

# Apply preprocessing to the 'description' column
cleaned_reviews = df['reviews'].dropna().apply(preprocess)
wordcloudrev1 = WordCloud(width=800, height=400, background_color='white').generate(' '.join(cleaned_reviews))
plt.figure(figsize=(10, 5))
plt.imshow(wordcloudrev1, interpolation='bilinear')
plt.axis('off')

plt.title("Word Cloud of Perfume Reviews")
plt.show()



In [17]:
#top rated vanilla perfumes

# Clean and process the notes column
df['notes'] = df['notes'].str.replace(r"\[|\]|'", "", regex=True)  # Remove brackets and apostrophes
df['notes'] = df['notes'].str.lower()  # Convert notes to lowercase for case-insensitive matching

# Filter for perfumes containing "vanilla" in the notes
vanilla_perfumes = df[df['notes'].str.contains('vanilla', na=False)]

# Sort by rating (descending order)
vanilla_perfumes_sorted = vanilla_perfumes.sort_values(by='rating', ascending=False)

# Display the top vanilla perfumes
print("Vanilla Perfumes Sorted by Rating:")
print(vanilla_perfumes_sorted[['notes', 'rating']].head(20))  # Display top 20 vanilla perfumes

# Save the results to a CSV file (optional)
vanilla_perfumes_sorted.to_csv('vanilla_perfumes_sorted_by_rating.csv', index=False)

print(vanilla_perfumes_sorted[['designer', 'title', 'notes', 'rating']].head(20))

plt.figure(figsize=(12, 6))
sns.barplot(x='rating', y='title', data=vanilla_perfumes_sorted.head(20), palette='viridis')
plt.title('Top Vanilla Perfumes by Rating')
plt.xlabel('Rating')
plt.ylabel('Title')
plt.show()

In [18]:
# sister notes
from itertools import combinations
from collections import defaultdict

# Assuming df is your DataFrame and it has a 'notes' column

# Clean and process the notes column
df['notes'] = df['notes'].str.replace(r"\[|\]|'", "", regex=True)  # Remove brackets and apostrophes
df['notes'] = df['notes'].str.lower()  # Convert notes to lowercase for consistency

# Split notes into lists and drop rows with missing notes
df['notes'] = df['notes'].str.split(',')
df = df.dropna(subset=['notes'])

# Create a co-occurrence matrix
co_occurrence = defaultdict(int)

# Iterate through each perfume's notes and count co-occurrences
for notes in df['notes']:
    # Generate all unique pairs of notes in the perfume
    unique_notes = list(set(notes))  # Remove duplicates within a perfume
    for note1, note2 in combinations(unique_notes, 2):
        # Sort the pair to avoid duplicate entries (e.g., (vanilla, musk) and (musk, vanilla))
        pair = tuple(sorted([note1.strip(), note2.strip()]))
        co_occurrence[pair] += 1

# Convert the co-occurrence dictionary to a DataFrame
co_occurrence_df = pd.DataFrame(list(co_occurrence.items()), columns=['Note Pair', 'Co-occurrence Count'])

# Sort by co-occurrence count (descending order)
co_occurrence_df = co_occurrence_df.sort_values(by='Co-occurrence Count', ascending=False)

# Display the top sister notes
print("Top Sister Notes (Most Frequently Co-occurring Pairs):")
print(co_occurrence_df.head(20))

# Save the results to a CSV file (optional)
co_occurrence_df.to_csv('sister_notes_co_occurrence.csv', index=False)

In [19]:
#let me see description embeddings data
df1 = pd.read_csv('/Users/sbayram/Downloads/chypre/description_embeddings.csv/data.csv')
df1.head()


In [20]:
import nbformat
from pathlib import Path

# Load the notebook
notebook_path = Path("/Users/sbayram/Downloads/chypre/frag_eda.ipynb")
with notebook_path.open() as f:
    nb = nbformat.read(f, as_version=4)

# Convert notebook cells to R Markdown
rmd_lines = ["---",
             "title: \"Fragrance EDA\"",
             "output: html_document",
             "---",
             ""]

for cell in nb.cells:
    if cell.cell_type == "markdown":
        rmd_lines.append(cell.source)
        rmd_lines.append("")
    elif cell.cell_type == "code":
        rmd_lines.append("```{r}")
        rmd_lines.append(cell.source)
        rmd_lines.append("```")
        rmd_lines.append("")

# Join all lines into a single string
rmd_content = "\n".join(rmd_lines)
rmd_path = "/Users/sbayram/Downloads/chypre/frag_eda.Rmd"

# Save to file
with open(rmd_path, "w") as f:
    f.write(rmd_content)

rmd_path